# Price Optimization Model

This notebook runs the modularized price optimization model.
All core functionality is in separate modules for maintainability.

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.optimize import minimize, NonlinearConstraint
import time

# Import our modularized functions
from data_processor import (
    load_and_filter_with_scope,
    pre_processor,
    create_elasticity_matrix,
    create_reference_arrays,
    create_bounds,
    create_segment_size_labels
)
from optimization import (
    create_evaluation_functions,
    create_objective_function,
    run_optimization,
    create_results_dataframe
)
from constraints import (
    create_constraint_functions,
    constraint_violation_detail,
    constraint_adherence,
    round_to_nearest_50
)

## User Inputs

Define optimization parameters and constraints.

In [ ]:
# USER INPUTS
target_pinc = 0.06      # Portfolio PINC
sku_lower_bound = -300
sku_upper_bound = 500
VAT = 0.19               # VAT for NR/Unit Calculation
VILC_GR = 0.0378         # VILC Annual Growth Rate

start_period = '2025-07'  # Optimization Starting Period
end_period = '2025-08'   # Optimizing Ending Period

# Penalty parameters
seg_lambda = 1e4
size_lambda = 1e4
penalty_per_violation = 1e3
tolerance = 0.01

# Hierarchy orders
segment_order = ["Value", "Core", "Core+", "Premium", "Super Premium"]
size_order = ["Small", "Regular", "Large"]

## Load Input Data

Load elasticity matrices and reference data.

In [ ]:
# READING IN ELASTICITY AND REFERENCE DATAFRAMES
base_path = r'C:/Users/40107922/OneDrive - Anheuser-Busch InBev/hackathon_2025/repo/hackathon_2025/optimization_data/'

sku_scope = pd.read_csv(base_path + 'sku_scope_subset.csv')
own_to_own_elasticity_df = pd.read_csv(base_path + 'elasticity.csv')
reference_df = pd.read_csv(base_path + 'reference_abi_sellin-vol_pl-ptc.csv')
own_to_competitor_elasticity_df = pd.read_csv(base_path + 'elasticity_competitor.csv')
competitor_reference_df = pd.read_csv(base_path + 'reference_comp_sellout-vol_ptc.csv')
seg_mapping = pd.read_csv(base_path + 'segment_mapping.csv')
sku_detail_mapping = pd.read_csv(base_path + 'sku_details_mapping.csv')

valid_periods = reference_df['year_month'].unique()
print(f"Loaded data with {len(valid_periods)} periods")

## Preprocess Data and Filter by Scope

Apply preprocessing and filter to SKU scope, calculating remaining ABI volume.

In [ ]:
# Load and filter data using the new function
(reference_df, remaining_abi_industry_volume, competitor_reference_df,
 own_to_own_elasticity_df, own_to_competitor_elasticity_df,
 reference_df_padded_bound, reference_df_other) = load_and_filter_with_scope(
    reference_df, sku_scope, VILC_GR,
    own_to_own_elasticity_df, own_to_competitor_elasticity_df,
    competitor_reference_df, seg_mapping, start_period, end_period,
    valid_periods
)

print(f"Remaining ABI industry volume: {remaining_abi_industry_volume:,.0f}")
print(f"Filtered to {len(reference_df['sku'].unique())} SKUs")

## Create Elasticity Matrices and Reference Arrays

Build the core data structures for optimization.

In [ ]:
# Create elasticity matrices
(own_products, competitor_products, months, num_months, num_own, num_comp, 
E_price_to_volume, E_price_to_comp_volume) = create_elasticity_matrix(
    own_to_own_elasticity_df, reference_df, 
    own_to_competitor_elasticity_df, competitor_reference_df
)

print(f"Own products: {num_own}, Competitor products: {num_comp}, Months: {num_months}")

# Create reference arrays
ref_arrays = create_reference_arrays(
    reference_df, competitor_reference_df, 
    months, own_products, competitor_products
)

# Unpack reference arrays
M, N, K = ref_arrays['M'], ref_arrays['N'], ref_arrays['K']
ref_price_liter = ref_arrays['ref_price_liter']
ref_price_unit = ref_arrays['ref_price_unit']
ref_vol_own_sellin = ref_arrays['ref_vol_own_sellin']
ref_vol_own_sellout = ref_arrays['ref_vol_own_sellout']
capacity = ref_arrays['capacity']
markup = ref_arrays['markup']
discount = ref_arrays['discount']
excise = ref_arrays['excise']
vilc = ref_arrays['vilc']
ref_price_comp = ref_arrays['ref_price_comp']
ref_vol_comp = ref_arrays['ref_vol_comp']

print(f"Reference arrays created: M={M}, N={N}, K={K}")

## Create Evaluation Functions with Caching

Set up the cached evaluation mechanism for fast optimization.

In [ ]:
# Create evaluation functions with caching
(evaluate_uncached, evaluate_cached, clear_eval_cache, 
 base_total_MACO, NR_ref, MACO_ref) = create_evaluation_functions(
    M, N, K, E_price_to_volume, E_price_to_comp_volume,
    ref_price_liter, ref_price_unit, ref_vol_own_sellin,
    ref_vol_own_sellout, ref_vol_comp, capacity,
    markup, discount, excise, vilc, VAT,
    remaining_abi_industry_volume
)

print(f"Baseline MACO: {base_total_MACO:,.2f}")

## Create Segment and Size Labels for Hierarchy Constraints

In [ ]:
# Create segment and size labels
segment_labels, size_labels = create_segment_size_labels(reference_df, M, N)
print(f"Segment labels shape: {segment_labels.shape}")
print(f"Size labels shape: {size_labels.shape}")

## Create Objective Function

In [ ]:
# Create objective function
objective = create_objective_function(
    evaluate_cached, segment_labels, size_labels,
    segment_order, size_order, seg_lambda, size_lambda,
    penalty_per_violation
)

print("Objective function created")

## Create Constraints

In [ ]:
# Create constraint functions
constraint_dict = create_constraint_functions(
    evaluate_cached, M, N, ref_price_liter, ref_vol_own_sellin,
    ref_vol_own_sellout, ref_vol_comp, remaining_abi_industry_volume,
    target_pinc, tolerance, base_total_MACO
)

nl_constraints = constraint_dict['nl_constraints']
TOTAL_REF_OWN_VOLUME = constraint_dict['TOTAL_REF_OWN_VOLUME']
TOTAL_REF_INDUSTRY_VOLUME = constraint_dict['TOTAL_REF_INDUSTRY_VOLUME']
REF_MARKET_SHARE = constraint_dict['REF_MARKET_SHARE']
REF_AVG_PRICE_LITER = constraint_dict['REF_AVG_PRICE_LITER']

print(f"Created {len(nl_constraints)} nonlinear constraints")

## Create Bounds

In [ ]:
# Create bounds
bounds = create_bounds(reference_df_padded_bound, sku_lower_bound, sku_upper_bound)
print(f"Created bounds for {len(bounds)} variables")

## Validate Bounds

Check if the upper bound can achieve the target PINC.

In [ ]:
# Set initial prices and validate bounds
P0 = ref_price_unit.reshape(-1)
p_unit_max = P0 + sku_upper_bound

test_eval = evaluate_uncached(p_unit_max)

if ((test_eval['avg_opt_price'] / REF_AVG_PRICE_LITER - 1) < target_pinc):
    print("❌ Error: Invalid PTC Bound Provided")
    print(f"Max achievable PINC: {(test_eval['avg_opt_price'] / REF_AVG_PRICE_LITER - 1):.4f}")
    print(f"Target PINC: {target_pinc:.4f}")
else:
    print("✅ Bounds are valid for target PINC")
    print(f"Max achievable PINC: {(test_eval['avg_opt_price'] / REF_AVG_PRICE_LITER - 1):.4f}")
    print(f"Target PINC: {target_pinc:.4f}")

## Run Optimization

Execute the trust-constr optimization with caching.

In [ ]:
# Clear cache before optimization
clear_eval_cache()

# Run optimization
res = run_optimization(
    objective, P0, bounds, nl_constraints,
    method='trust-constr', maxiter=5000, disp=True
)

print(f"\nOptimization status: {res.success}")
print(f"Message: {res.message}")

## Check Constraint Violations

In [ ]:
# Check constraint violations
print("\nConstraint Violation Details:")
for idx, val, lb, ub, viol in constraint_violation_detail(res.x, nl_constraints):
    print(f"Constraint {idx}: value={val:.6g}, lb={lb:.6g}, ub={ub:.6g}, violation={viol:.6g}")

## Round Prices and Validate

Round prices to nearest 50 and check constraint adherence.

In [ ]:
# Round prices
P_opt_final = res.x.reshape(M, N)
P_rounded = round_to_nearest_50(res.x, P0)
P_rounded = P_rounded.reshape(M, N)

print("="*80)
print("ORIGINAL METRICS (Before Rounding):")
print("="*80)
constraint_adherence(
    P_opt_final, evaluate_uncached, base_total_MACO, TOTAL_REF_OWN_VOLUME,
    TOTAL_REF_INDUSTRY_VOLUME, REF_MARKET_SHARE, target_pinc, tolerance,
    segment_labels, size_labels, segment_order, size_order, M, N
)

print("\n" + "="*80)
print("POST ROUNDING METRICS:")
print("="*80)
constraint_adherence(
    P_rounded, evaluate_uncached, base_total_MACO, TOTAL_REF_OWN_VOLUME,
    TOTAL_REF_INDUSTRY_VOLUME, REF_MARKET_SHARE, target_pinc, tolerance,
    segment_labels, size_labels, segment_order, size_order, M, N
)

## Create Results DataFrame

In [ ]:
# Create final evaluation with rounded prices
final_eval = evaluate_uncached(P_rounded.reshape(-1))

# Create results dataframes (both monthly_outputs and industry_volumes)
results_df, industry_df = create_results_dataframe(
    P_rounded, reference_df, seg_mapping, sku_detail_mapping,
    ref_price_liter, ref_price_unit, ref_vol_own_sellin,
    NR_ref, MACO_ref, capacity, final_eval, M, N,
    ref_vol_own_sellout, competitor_reference_df, reference_df_other
)

print(f"\nResults dataframe created with {len(results_df)} rows")
print(f"Industry dataframe created with {len(industry_df)} rows")
results_df.head()

In [ ]:
industry_df.manufacturer.unique()

## Save Results

In [ ]:
# Save results
output_path = 'optimization_results/optimization_output.xlsx'
results_df.to_excel(output_path, index=False)
print(f"Results saved to: {output_path}")